# Data Exploration & Analysis

Before you build anything, you have to understand your materials. When I was
designing circuits, I'd always characterize my components first — measure the
tolerances, test the edge cases, know what I'm working with. Same principle here.

We've got a deliberately heterogeneous corpus — three very different text sources
that'll stress-test our extraction and summarization pipelines in different ways:

- **Amazon Product Reviews** (~74k) — short, opinionated, sentiment-heavy
- **Customer Support Tickets** (~29.8k) — template-heavy, noisy, issue-focused
- **CNN/DailyMail News** (~11.5k) — long-form, entity-dense, with gold-standard summaries

Let's poke around and see what we're dealing with.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data.loader import load_reviews, load_support_tickets, load_news
from src.data.preprocessing import preprocess_documents, get_text_stats, tokenize_words

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('All imports loaded — let\'s go!')

## 1. Load All Datasets

Each loader normalizes its source into our universal Document format.
Different input formats, same output structure — that's the whole point.

In [ ]:
# Load each source separately so we can analyze per-source characteristics
reviews = load_reviews(max_docs=5000)
tickets = load_support_tickets(max_docs=5000)

# CNN/DailyMail downloads on first run (~300MB). Caches after that.
news = load_news(max_docs=2000)

print(f'Reviews: {len(reviews)}')
print(f'Tickets: {len(tickets)}')
print(f'News:    {len(news)}')

In [ ]:
# Run preprocessing and compute text stats for EDA
all_docs = reviews + tickets + news
all_docs = preprocess_documents(all_docs, compute_stats=True)

# Flatten into a DataFrame — pandas is great for exploratory slicing
rows = []
for doc in all_docs:
    row = {
        'id': doc.id,
        'source_type': doc.source_type,
        'text': doc.text,
        'created_at': doc.created_at,
        **doc.metadata.get('text_stats', {}),
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

## 2. Corpus Overview

First impressions matter. Let's see the shape of each source —
how many documents, how long they are, how complex the sentences.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# How many documents per source?
source_counts = df['source_type'].value_counts()
axes[0].bar(source_counts.index, source_counts.values, color=['#4285F4', '#EA4335', '#34A853'])
axes[0].set_title('Documents by Source')
axes[0].set_ylabel('Count')

# How long are they? (word count distribution)
for src in df['source_type'].unique():
    subset = df[df['source_type'] == src]
    axes[1].hist(subset['word_count'].clip(upper=500), bins=50, alpha=0.6, label=src)
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].legend()

# How complex are the sentences?
avg_sent = df.groupby('source_type')['avg_sentence_length'].mean()
axes[2].bar(avg_sent.index, avg_sent.values, color=['#4285F4', '#EA4335', '#34A853'])
axes[2].set_title('Avg Sentence Length')
axes[2].set_ylabel('Words per Sentence')

plt.tight_layout()
plt.show()

In [ ]:
# The numbers behind the pictures
stats_summary = df.groupby('source_type')[['char_count', 'word_count', 'sentence_count', 'avg_sentence_length', 'stopword_ratio']].describe()
stats_summary.round(2)

## 3. Per-Source Deep Dive

### 3a. Amazon Reviews

Reviews are the heart of sentiment analysis. Short texts, strong opinions,
product and brand entities everywhere. Let's see what we're working with.

In [ ]:
review_meta = pd.DataFrame([d.metadata for d in reviews])
review_meta['text_length'] = [len(d.text) for d in reviews]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Star ratings — are we dealing with mostly happy or mostly angry customers?
if 'rating' in review_meta.columns:
    rating_counts = review_meta['rating'].value_counts().sort_index()
    axes[0].bar(rating_counts.index.astype(str), rating_counts.values, color='#4285F4')
    axes[0].set_title('Review Rating Distribution')
    axes[0].set_xlabel('Rating')

# Which brands show up the most?
if 'brand' in review_meta.columns:
    top_brands = review_meta['brand'].value_counts().head(10)
    axes[1].barh(top_brands.index, top_brands.values, color='#4285F4')
    axes[1].set_title('Top 10 Brands')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 3b. Support Tickets

These are noisy and template-heavy, but the metadata is rich —
ticket types, priorities, products, resolution status. The text itself
needs aggressive preprocessing (those {placeholder} tags!), but the
signal is there if you clean it up.

In [ ]:
ticket_meta = pd.DataFrame([d.metadata for d in tickets])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# What kinds of tickets are people filing?
if 'type' in ticket_meta.columns:
    type_counts = ticket_meta['type'].value_counts().head(8)
    axes[0].barh(type_counts.index, type_counts.values, color='#EA4335')
    axes[0].set_title('Ticket Types')
    axes[0].invert_yaxis()

# Priority distribution — are most tickets critical or low?
if 'priority' in ticket_meta.columns:
    prio_counts = ticket_meta['priority'].value_counts()
    axes[1].bar(prio_counts.index, prio_counts.values, color='#EA4335')
    axes[1].set_title('Ticket Priority Distribution')

plt.tight_layout()
plt.show()

## 4. Cross-Source Vocabulary Analysis

Different sources use different words. Reviews talk about "battery" and
"quality", tickets talk about "issue" and "support", news talks about
"said" and "government". Understanding these vocabulary differences
tells us a lot about what our extraction pipeline needs to handle.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (src, docs) in enumerate([
    ('review', reviews[:2000]),
    ('support_ticket', tickets[:2000]),
    ('news', news[:2000]),
]):
    all_words = []
    for doc in docs:
        all_words.extend(tokenize_words(doc.text, remove_stopwords=True))

    top_20 = Counter(all_words).most_common(20)
    words, counts = zip(*top_20)

    colors = ['#4285F4', '#EA4335', '#34A853']
    axes[i].barh(list(words), list(counts), color=colors[i])
    axes[i].set_title(f'Top 20 Words: {src}')
    axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Sample Documents

Nothing beats eyeballing the actual data. Here's a representative
sample from each source. Look at the text quality, the metadata
richness, the formatting quirks.

In [ ]:
import textwrap
import json

for src, docs in [('Review', reviews), ('Ticket', tickets), ('News', news)]:
    print(f'\n{"="*80}')
    print(f' {src} Sample')
    print(f'{"="*80}')
    doc = docs[0]
    print(f'ID: {doc.id}')
    print(f'Text: {textwrap.shorten(doc.text, 300)}')
    print(f'Metadata: {json.dumps({k: v for k, v in doc.metadata.items() if k != "text_stats"}, indent=2, default=str)}')

## 6. Key Takeaways

Here's what we know about our materials now:

- **Reviews** are short-to-medium, sentiment-heavy, packed with product/brand entities. Great for sentiment analysis and opinion mining.
- **Support tickets** are short, template-heavy with lots of noise, but carry rich metadata (priority, product, resolution). The preprocessing pipeline earns its keep here.
- **News articles** are long-form, entity-dense (people, orgs, locations, dates), and come with gold-standard human summaries we can evaluate against.
- The three sources differ dramatically in vocabulary, structure, and length — exactly the kind of variation that stress-tests a pipeline and proves it generalizes.

Now that we know what we're working with, it's time to run the extraction
and summarization pipelines and see how well they perform. Head to
**notebook 02** for that.